# Stage 6 — Theory of Mind Reasoning Trace Verifier

**Pipeline position:** Stage 5 generates two parallel reasoning trace sets
(`reasoning_react.csv` from ReAct framework, `reasoning_recon.csv` from ReCon
framework). This Stage 6 notebook verifies and selects the best trace per
game/round using Claude Sonnet 4.5 as a blind pairwise judge.

**Approach:** Structured multi-criteria blind binary scoring (Approach 2a).
- 4 binary criteria: Abduction Plausibility, Suspicion Level Consistency,
  Gated Depth Reasoning, Statement & Deduction Coherence.
- Blind A/B presentation (Trace A = ReAct, Trace B = ReCon; not labeled).
- Layer 1: pairwise comparison → winner selection or targeted correction.
- Layer 2: inline recheck after targeted correction (up to 3 attempts).
- No custom generation — always fall back to the better trace.
- Player roles REVEALED to verifier for Definitive Deduction Rule validation.

**Outputs:**
- `Datasets/verified/reasoning_gold_verified.csv`
- `Datasets/verified/reasoning_gold_criteria_scores.csv`


In [16]:
import os, json, time, ast, random
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ── Paths ──────────────────────────────────────────────────────────────────
REACT_CSV_PATH  = Path('Datasets/reasoning/reasoning_react.csv')
RECON_CSV_PATH  = Path('Datasets/reasoning/reasoning_recon.csv')
OUTPUT_DIR      = Path('Datasets/verified')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VERIFIED_CSV = OUTPUT_DIR / 'reasoning_gold_verified.csv'
OUTPUT_CRITERIA_CSV = OUTPUT_DIR / 'reasoning_gold_criteria_scores.csv'

# ── Configuration ──────────────────────────────────────────────────────────
VERIFIER_MODEL      = 'claude-sonnet-4-6'
MAX_API_RETRIES     = 5
SAVE_EVERY_N_ROWS   = 10   # flush to disk every N rows
TEST_LIMIT          = 2  # set to small int (e.g. 2) for test runs; None = all

# 9 source columns preserved verbatim in the output CSV
PRESERVE_COLS = [
    'game_id', 'round_id', 'role_id', 'llm_alignment',
    'player_roles', 'public_history', 'prior_summary_gold',
    'discussion_log', 'matrix_tactic_scale',
]

# ── Sanity-check source CSVs ───────────────────────────────────────────────
for _p, _name in [(REACT_CSV_PATH, 'reasoning_react.csv'),
                   (RECON_CSV_PATH, 'reasoning_recon.csv')]:
    if _p.exists():
        print(f'✓ {_name} found  ({_p})')
    else:
        print(f'⚠  {_name} NOT FOUND at {_p} — run log-gen-reasoner.ipynb first')

print(f'\nVerifier model : {VERIFIER_MODEL}')
print(f'Save cadence   : every {SAVE_EVERY_N_ROWS} rows')
print(f'Test limit     : {TEST_LIMIT or "all rows"}')


✓ reasoning_react.csv found  (Datasets\reasoning\reasoning_react.csv)
✓ reasoning_recon.csv found  (Datasets\reasoning\reasoning_recon.csv)

Verifier model : claude-sonnet-4-6
Save cadence   : every 10 rows
Test limit     : 2


## Section 2: Load Datasets and Sanity Checks


In [ ]:
react_df = pd.read_csv(REACT_CSV_PATH)
recon_df = pd.read_csv(RECON_CSV_PATH)

# Sort both by game_id then round_id (numeric order)
def _sort_key_df(df):
    df = df.copy()
    df['_gn'] = df['game_id'].str.replace('G', '', regex=False).astype(int)
    df['_rn'] = df['round_id'].astype(str).str.replace('R', '', regex=False).astype(int)
    df = df.sort_values(['_gn', '_rn']).drop(columns=['_gn', '_rn'])
    return df.reset_index(drop=True)

react_df = _sort_key_df(react_df)
recon_df = _sort_key_df(recon_df)

print(f'ReAct CSV : {react_df.shape[0]} rows, {react_df.shape[1]} cols')
print(f'ReCon CSV : {recon_df.shape[0]} rows, {recon_df.shape[1]} cols')

# Verify ID alignment
react_ids = list(zip(react_df['game_id'].astype(str), react_df['round_id'].astype(str)))
recon_ids = list(zip(recon_df['game_id'].astype(str), recon_df['round_id'].astype(str)))
if react_ids == recon_ids:
    print(f'✓ Both CSVs contain the same {len(react_ids)} game/round pairs in the same order')
else:
    missing_in_recon = set(react_ids) - set(recon_ids)
    missing_in_react = set(recon_ids) - set(react_ids)
    print(f'⚠  Alignment mismatch: {len(missing_in_recon)} in ReAct but not ReCon; '
          f'{len(missing_in_react)} in ReCon but not ReAct')
    print('  Proceeding with intersection-based lookup.')

# Build recon lookup  (game_id, round_id) -> Series
recon_lookup = {}
for _, _row in recon_df.iterrows():
    recon_lookup[(str(_row['game_id']), str(_row['round_id']))] = _row

print(f'\n✓ recon_lookup built: {len(recon_lookup)} entries')

# ── Build prior summaries map from summarizer CSVs ───────────────────────────
SUMMARY_DIR = Path('Datasets/summarizer')
_prior_summary_map = {}   # {game_id: {round_id_str: summary_text}}

_summary_files = []
if SUMMARY_DIR.exists():
    for _fp in SUMMARY_DIR.glob('summaries_r*.csv'):
        _stem = _fp.stem.lower()  # e.g., summaries_r3
        if not _stem.startswith('summaries_r'):
            continue
        _tail = _stem.replace('summaries_r', '', 1)
        if _tail.isdigit():
            _summary_files.append((int(_tail), _fp))

_summary_files = sorted(_summary_files, key=lambda x: x[0])
_summary_rows_loaded = 0

for _rnum, _fp in _summary_files:
    _sdf = pd.read_csv(_fp)
    if _sdf.empty:
        continue

    _colmap = {str(c).strip().lower(): c for c in _sdf.columns}
    _gid_col = _colmap.get('game_id')
    _rid_col = _colmap.get('round_id')
    _sum_col = _colmap.get('round_summary') or _colmap.get('summary')

    if not _gid_col or not _rid_col or not _sum_col:
        print(f'⚠  Skipping {_fp.name}: required columns missing')
        continue

    for _, _srow in _sdf.iterrows():
        _gid = str(_srow[_gid_col]).strip()
        if not _gid or _gid.lower() == 'nan':
            continue

        _rid_raw = str(_srow[_rid_col]).strip()
        try:
            _rid_num = int(_rid_raw.replace('R', '').replace('r', ''))
        except Exception:
            continue

        _txt = _srow.get(_sum_col, '')
        if pd.isna(_txt):
            _txt = ''
        _txt = str(_txt).strip()
        if not _txt:
            continue

        if _gid not in _prior_summary_map:
            _prior_summary_map[_gid] = {}
        _prior_summary_map[_gid]['R' + str(_rid_num)] = _txt
        _summary_rows_loaded += 1

print(f'\n✓ Prior summaries indexed from summarizer CSVs: '
      f'{len(_prior_summary_map)} games, {_summary_rows_loaded} round summaries')

# ── Load prior reasoning golds from previous runs of this notebook ──────────
# Stored as dict-of-dict to avoid duplicate round entries on repeated reruns.
_prior_gold_map = {}   # {game_id: {round_id_str: reasoning_gold_str}}
if OUTPUT_VERIFIED_CSV.exists():
    _vdf = pd.read_csv(OUTPUT_VERIFIED_CSV)
    for _, _vrow in _vdf.iterrows():
        _gid  = str(_vrow['game_id'])
        _rid  = str(_vrow['round_id'])
        _gold = _vrow.get('reasoning_gold', '')
        if pd.isna(_gold):
            _gold = ''
        _gold = str(_gold).strip()
        if _gold:
            if _gid not in _prior_gold_map:
                _prior_gold_map[_gid] = {}
            _prior_gold_map[_gid][_rid] = _gold
    print(f'\n✓ Prior reasoning golds loaded: {len(_prior_gold_map)} games '
          f'from {OUTPUT_VERIFIED_CSV.name}')
else:
    print('\n⚠  No prior reasoning golds found (output CSV not found — starting fresh)')

# ── Build ReCon fallback map for cascading gold gaps ─────────────────────────
# When a prior round fails to produce a verified gold (API error, parse failure,
# needs_retry with no Step-7b recovery), _prior_gold_map won't have that round.
# get_prior_gold_str will fall back to the raw ReCon trace so downstream rounds
# still receive meaningful prior-round context rather than an empty string.
_recon_fallback_map = {}   # {game_id: {round_id_str: recon_trace_str}}
_recon_gold_col = 'reasoning_gold_recon'
if _recon_gold_col in recon_df.columns:
    for _, _rrow in recon_df.iterrows():
        _gid  = str(_rrow['game_id']).strip()
        _rid  = str(_rrow['round_id']).strip()
        _rtrc = _rrow.get(_recon_gold_col, '')
        if pd.isna(_rtrc):
            _rtrc = ''
        _rtrc = str(_rtrc).strip()
        if _gid and _rid and _rtrc:
            if _gid not in _recon_fallback_map:
                _recon_fallback_map[_gid] = {}
            _recon_fallback_map[_gid][_rid] = _rtrc
    print(f'\n✓ ReCon fallback map built: {len(_recon_fallback_map)} games '
          f'({sum(len(v) for v in _recon_fallback_map.values())} rounds)')
else:
    print(f'\n⚠  Column "{_recon_gold_col}" not found in recon_df — ReCon gold fallback disabled')

print(f'\nSample row (ReAct):\n', react_df.iloc[0][PRESERVE_COLS + ['reasoning_gold_react']].to_dict())


ReAct CSV : 1000 rows, 10 cols
ReCon CSV : 1000 rows, 10 cols
✓ Both CSVs contain the same 1000 game/round pairs in the same order

✓ recon_lookup built: 1000 entries

✓ Prior summaries indexed from summarizer CSVs: 250 games, 1000 round summaries

⚠  No prior reasoning golds found (output CSV not found — starting fresh)

Sample row (ReAct):
 {'game_id': 'G001', 'round_id': 1, 'role_id': 'P1', 'llm_alignment': 'Good', 'player_roles': '{"P1":"Good",\n  "P2":"Good",\n  "P3":"Evil",\n  "P4":"Evil",\n  "P5":"Good"}', 'public_history': 'Round: 1\nLeader: P4\nTeam: P1, P4\nVotes: P1:Y P2:Y P3:Y P4:Y P5:Y\nQuest 1 Outcome: FAIL', 'prior_summary_gold': '""', 'discussion_log': 'Discussion after Quest 1:\n\nP3: "I think that P1, P4, and me would make a great team"\n\nP2: "P1 and P4 went on the last mission and it failed. P3 must be evil"\n\nP4: "I know the first mission with me and P1 failed, but it was obviously P1 who failed it"\n\nP5: "It is not a good idea to form a team with both P1 and P4.

## Section 3: Context Builder Helpers


In [ ]:
# ── Protagonist assignment (same logic as log-gen-reasoner.ipynb) ──────────
def get_protagonist(game_id: str) -> str:
    num = int(str(game_id).replace('G', ''))
    if num <= 50:  return 'P1'
    if num <= 100: return 'P2'
    if num <= 150: return 'P3'
    if num <= 200: return 'P4'
    return 'P5'

def resolve_protagonist(game_id: str, row: pd.Series) -> str:
    """Prefer role_id from row when available; fallback to deterministic mapping."""
    rid = str(row.get('role_id', '') or '').strip().upper()
    if rid in {'P1', 'P2', 'P3', 'P4', 'P5'}:
        return rid
    return get_protagonist(game_id)


_TRACE_REQUIRED_KEYS = {'abduction', 'suspicion', 'depth', 'beliefs', 'statement', 'deduction'}

def validate_full_trace_object(trace_obj) -> bool:
    """Light schema check for a full corrected trace payload."""
    if not isinstance(trace_obj, dict):
        return False
    if not _TRACE_REQUIRED_KEYS.issubset(set(trace_obj.keys())):
        return False
    if not isinstance(trace_obj.get('abduction'), list):
        return False
    if not isinstance(trace_obj.get('suspicion'), list):
        return False
    if not isinstance(trace_obj.get('beliefs'), list):
        return False
    if not isinstance(trace_obj.get('statement'), str):
        return False
    if not isinstance(trace_obj.get('deduction'), list):
        return False
    return True

# ── Round sort key ──────────────────────────────────────────────────────────
def _round_sort_key(r) -> int:
    try:
        return int(str(r).replace('R', '').replace('r', ''))
    except:
        return 999

# ── Cumulative context builder ──────────────────────────────────────────────
def build_cumulative_context(game_id: str, round_id: str, df: pd.DataFrame) -> dict:
    """
    Reconstruct cumulative public_history and discussion_log up to and
    including `round_id` for `game_id`, from a per-round reasoning CSV.
    Also determine is_final_round flag.
    """
    game_rows = df[df['game_id'].astype(str) == str(game_id)].copy()
    game_rows['_rnum'] = game_rows['round_id'].apply(_round_sort_key)
    game_rows = game_rows.sort_values('_rnum')

    target_rnum = _round_sort_key(round_id)
    cum_hist, cum_disc = [], []

    for _, row in game_rows.iterrows():
        r = str(row['round_id'])
        rnum = _round_sort_key(r)
        if rnum > target_rnum:
            break
        r_label = r if r.upper().startswith('R') else 'R' + r
        h = str(row.get('public_history', '') or '').strip()
        d = str(row.get('discussion_log', '') or '').strip()
        if h:
            cum_hist.append('[' + r_label + '] ' + h)
        if d:
            cum_disc.append('[' + r_label + '] ' + d)

    all_rnums = sorted(game_rows['_rnum'].tolist())
    is_final = (target_rnum == all_rnums[-1]) if all_rnums else False

    return {
        'cumulative_public_history': '\n\n'.join(cum_hist),
        'cumulative_discussion_log': '\n\n'.join(cum_disc),
        'is_final_round': is_final,
    }

# ── Prior summary helper ───────────────────────────────────────────────────
def get_prior_summary_str(game_id: str, round_id: str) -> str:
    """
    Returns cumulative round summaries for all rounds up to and including round_id.
    Uses _prior_summary_map from global scope.
    """
    target_rnum = _round_sort_key(round_id)
    parts = []
    for rid, summary in sorted(
        _prior_summary_map.get(game_id, {}).items(),
        key=lambda x: _round_sort_key(x[0])
    ):
        if _round_sort_key(rid) <= target_rnum:
            sval = str(summary or '').strip()
            if sval:
                parts.append(sval)
    return '\n\n'.join(parts) if parts else ''

# ── Prior reasoning gold helper ─────────────────────────────────────────────
def get_prior_gold_str(game_id: str, round_id: str) -> str:
    """
    Returns verified reasoning golds for all rounds numerically before round_id.
    Primary source: _prior_gold_map (populated from output CSV + in-run updates).
    Fallback source: _recon_fallback_map (raw ReCon traces) — used when a prior
    round has no verified gold due to API error, parse failure, or retry exhaustion.
    Fallback entries are labelled "(ReCon fallback — unverified)" so the verifier
    can weight them appropriately.
    """
    target_rnum = _round_sort_key(round_id)
    game_gold     = _prior_gold_map.get(game_id, {})
    game_fallback = _recon_fallback_map.get(game_id, {})

    # Collect all known prior round IDs from both maps (union)
    all_prior_rids = {
        rid for rid in (set(game_gold.keys()) | set(game_fallback.keys()))
        if _round_sort_key(rid) < target_rnum
    }

    parts = []
    for rid in sorted(all_prior_rids, key=_round_sort_key):
        r_label = rid if str(rid).upper().startswith('R') else 'R' + str(rid)
        gval = str(game_gold.get(rid, '') or '').strip()
        if gval:
            parts.append(f'[{r_label}]\n{gval}')
        else:
            fallback = str(game_fallback.get(rid, '') or '').strip()
            if fallback:
                parts.append(f'[{r_label}] (ReCon fallback — unverified)\n{fallback}')
    return '\n\n'.join(parts) if parts else ''

# ── Trace string parser ─────────────────────────────────────────────────────
def parse_trace_string(trace_str) -> dict:
    """Parse a stored reasoning trace string (JSON or ast fallback)."""
    if not trace_str or str(trace_str).strip() in ('', 'nan', 'None'):
        return {}
    try:
        return json.loads(str(trace_str).strip())
    except Exception:
        try:
            return ast.literal_eval(str(trace_str).strip())
        except Exception:
            return {}

# ── Trace JSON formatter (matches log-gen-reasoner.ipynb output) ────────────
def format_trace_json(trace: dict) -> str:
    """Serialize a reasoning trace with top-level keys on separate lines."""
    if not trace:
        return '{}'
    lines = ['{']
    items = list(trace.items())
    for i, (key, value) in enumerate(items):
        comma = ',' if i < len(items) - 1 else ''
        if isinstance(value, list) and value and isinstance(value[0], dict):
            lines.append(f'  "{key}": [')
            for j, elem in enumerate(value):
                elem_comma = ',' if j < len(value) - 1 else ''
                lines.append('    ' + json.dumps(elem, ensure_ascii=False,
                              separators=(',', ':')) + elem_comma)
            lines.append('  ]' + comma)
        else:
            lines.append('  "{key}": {val}{comma}'.format(
                key=key,
                val=json.dumps(value, ensure_ascii=False, separators=(',', ':')),
                comma=comma))
    lines.append('}')
    return '\n'.join(lines)

print('✓ Helper functions defined: get_protagonist, resolve_protagonist,')
print('  validate_full_trace_object, build_cumulative_context,')
print('  get_prior_summary_str, get_prior_gold_str, parse_trace_string, format_trace_json')


✓ Helper functions defined: get_protagonist, resolve_protagonist,
  validate_full_trace_object, build_cumulative_context,
  get_prior_summary_str, get_prior_gold_str, parse_trace_string, format_trace_json


## Section 4: Claude Verifier Initialization


In [20]:
# ── Claude Sonnet 4.5 client (Anthropic OpenAI-compatible endpoint) ─────────
_api_key = os.getenv('ANTHROPIC_API_KEY')
if not _api_key:
    raise EnvironmentError('ANTHROPIC_API_KEY not set in environment / .env file')

client = OpenAI(
    api_key=_api_key,
    base_url='https://api.anthropic.com/v1/',
)

# ── Criteria and label constants ────────────────────────────────────────────
_CRITERIA_ORDER    = ['abduction', 'suspicion', 'depth', 'statement']
_FRAMEWORK_LABEL   = {
    'TRACE_A':      'ReAct',
    'CORRECTION_A': 'ReAct-Claude-4.5',
    'TRACE_B':      'ReCon',
    'CORRECTION_B': 'ReCon-Claude-4.5',
}
_PRINT_PROMPT_DEBUG = {'layer1': True, 'layer2': False}
ENABLE_RECOMMENDATION_SAFETY_OVERRIDE = False

# ── System prompt ────────────────────────────────────────────────────────────
VERIFIER_SYSTEM_PROMPT = (
    'You are an expert verifier for Avalon Theory of Mind reasoning traces. '
    'Evaluate TRACE A and TRACE B using 4 binary PASS/FAIL criteria, apply the '
    'decision logic strictly, and return only valid JSON. Judge from the '
    'Investigator\'s perspective using observable evidence only, not factual '
    'alignment truth. Only fail a criterion when the violation is clear.'
)

# ── Avalon mechanics reference ───────────────────────────────────────────────
AVALON_RULES_REF = '''5-player game: 3 Good players, 2 Evil players. Evil players know each other.
Quest team sizes: R1=2, R2=3, R3=2, R4=3, R5=3.
All team proposals in this dataset were approved unanimously.
Quest outcomes: one FAIL card fails the quest. Good players ALWAYS submit PASS.

INVESTIGATOR ROLE
- The Investigator is ALWAYS Good.
- The Investigator ALWAYS submitted PASS on every quest.
- The Investigator does NOT speak in the discussion log.
- The Investigator reasons only from observable evidence: quest outcomes, team
  compositions, votes, and discussion statements.

DEFINITIVE DEDUCTION RULE
- 2-person failed quest with Investigator on team: the other team member is definitively Evil.
  REQUIRED: choice="Evil", suspicion="Extreme", and included in deduction.
- 3-person failed quest with Investigator on team: at least one of the other two is Evil, but identity is uncertain.
  Expected: at least Moderate suspicion with hedged reasoning.
- Investigator not on team: infer only from team compositions, voting, outcomes, and discussion.

TRACE FIELDS
- abduction: 4 entries, one per non-investigator player
- suspicion: 4 entries, same player order
- depth: 1 or 2
- beliefs: level-1 for all 4 players; level-2 only when gated by suspicion
- statement: 1-2 sentence in-character dialogue
- deduction: 1-3 players, except final round = exactly 2

GATED DEPTH RULE
- depth=2 iff at least one player has Strong or Extreme suspicion.
- level-2 beliefs exist ONLY for Strong/Extreme players.
- level-2 content must be genuine second-order reasoning.

SUSPICION SCALE
Minimal < Mild < Moderate < Strong < Extreme'''

# ── Criteria definitions ─────────────────────────────────────────────────────
CRITERIA_DEFINITIONS_REF = '''CRITERION 1 -- ABDUCTION PLAUSIBILITY (1=PASS, 0=FAIL)
Abduction is the Investigator's inference about each player's alignment from observable evidence. 
Any combination of choices is valid: 0, 1, 2, or 3 Evil choices.

PASS if and only if ALL of the following hold:
  - Each player's good_expl and evil_expl are distinct, cite the strongest available observable evidence from the discussion log, and do not substitute generic behavioral reads when more specific grounding exists.
  - The explanations fit the discussion log and public evidence.
  - The choice reflects the strongest alignment inference from cumulative public evidence.
  - If the evidence is genuinely ambiguous, either choice may pass only if the explanation cites specific evidence, acknowledges a competing read of roughly equal weight, and presents the choice tentatively rather than as the dominant inference.
  - DEFINITIVE DEDUCTION RULE: a 2-person failed quest partner must have choice="Evil".
FAIL if ANY of the following hold:
  - good_expl and evil_expl, templated, or do not fit the discussion log.
  - good_expl and evil_expl are nearly identical.
  - Definitive Deduction Rule is violated.
  - The choice relies on a weaker cue while stronger public evidence points elsewhere.
  - The choice overstates confidence for an ambiguous case, especially when only a single cue is present.
NOTE ON GROUND TRUTH ROLES: Use ground truth only as a sanity check for clear Definitive Deduction Rule violations.
Do NOT fail this criterion only because the choice disagrees with ground truth.

CRITERION 2 -- SUSPICION LEVEL CONSISTENCY (1=PASS, 0=FAIL)
Suspicion is scaffolded from abduction plus observable evidence.
Flow: abduction choice -> suspicion level.

PASS if and only if ALL of the following hold:
  - Levels are proportionate to the observable evidence.
  - DEFINITIVE DEDUCTION RULE: a 2-person failed quest partner must have level="Extreme".
  - In ambiguous cases, Mild or Moderate suspicion is acceptable and should stay hedged.
  - Strong or Extreme requires multiple converging public cues or a definitive deduction rule.
  - A player with choice="Evil" has level >= Moderate.
  - In R3+, players on multiple failed quests have at least Moderate suspicion.
FAIL if ANY of the following hold:
  - Strong or Extreme without multiple converging public cues or a definitive deduction rule.
  - Strong/Extreme is used for a genuinely mixed case that only supports a tentative read.
  - Strong/Extreme is assigned with no supporting observable evidence.
  - A player with choice="Evil" has Minimal or Mild suspicion.
  - A definitively Evil player has less than Extreme suspicion.
  - In R3+, a player on multiple failed quests has only Minimal/Mild suspicion.

CRITERION 3 -- GATED DEPTH REASONING (1=PASS, 0=FAIL)
Depth is determined mechanically from the actual suspicion levels written in the suspicion array. 
Do not re-derive suspicion from abduction while scoring depth.

PASS if and only if ALL of the following hold:
  - depth=2 iff at least one player has Strong or Extreme suspicion; otherwise depth=1.
  - Exactly 4 level-1 beliefs exist.
  - level-2 beliefs exist for ALL Strong/Extreme players and for NO Minimal/Mild/Moderate players.
  - level-2 content is genuine second-order reasoning, not a restatement of level-1.
FAIL if ANY of the following hold:
  - depth=1 despite a Strong/Extreme suspicion entry.
  - depth=2 but no level-2 entries exist.
  - A Minimal/Mild/Moderate player has a level-2 belief entry.
  - level-2 content is substantively identical to level-1.

CRITERION 4 -- STATEMENT AND DEDUCTION COHERENCE (1=PASS, 0=FAIL)
Judge naturalness from the Investigator's perspective using observable evidence only.
The statement and deduction do not need to be factually correct about actual alignments.

PASS if and only if ALL of the following hold:
  - Statement sounds like believable Avalon dialogue and is grounded in public evidence.
  - Deduction follows naturally from abduction, suspicion, and the observed evidence.
  - Deduction includes ALL Strong/Extreme suspicion players.
  - FINAL ROUND: deduction contains exactly 2 players.
  - NON-FINAL: deduction contains 1-3 players.
FAIL if ANY of the following hold:
  - Deduction omits a Strong/Extreme suspicion player.
  - Final round deduction does not contain exactly 2 players.
  - Deduction clearly contradicts the trace's own abduction/suspicion/depth.
  - Statement uses information not observable in the public history or discussion.'''

# ── Decision logic reference ─────────────────────────────────────────────────
DECISION_LOGIC_REF = '''
DECISION LOGIC
1. Both traces 4/4 -> PAIRWISE_TIEBREAKER
   - Choose the slightly stronger trace overall which is closer to the ground-truth player alignment.
   - Set pairwise_winner to TRACE_A or TRACE_B.
   - Set pairwise_reasoning to a brief explanation.
   - failed_criteria = null, targeted_correction = null.

2. One trace 4/4, the other <4/4 -> TRACE_A or TRACE_B
   - Select the 4/4 trace as-is.
   - pairwise_winner = null, pairwise_reasoning = null.
   - failed_criteria = null, targeted_correction = null.

3. Both traces <4/4 -> CORRECTION_A or CORRECTION_B
   - If scores differ, choose the higher-scoring trace as the base.
   - If scores tie, choose the stronger base holistically.
   - targeted_correction must correct the chosen base trace.
   - Fix all failed criteria and keep the full trace internally consistent.
   
HARD RULES 
- CORRECTION_A or CORRECTION_B is allowed only if BOTH traces score <4/4.
- If TRACE A scores 4/4 and TRACE B scores <4/4, recommendation must be TRACE_A.
- If TRACE B scores 4/4 and TRACE A scores <4/4, recommendation must be TRACE_B.
- If both traces score 4/4, recommendation must be PAIRWISE_TIEBREAKER.'''

# ── Evaluation process reference ─────────────────────────────────────────────
EVALUATION_PROCESS_REF = '''1. Evaluate TRACE A against all 4 criteria using binary PASS/FAIL checks.
2. Evaluate TRACE B against all 4 criteria using the same binary checks.
3. Count total passes for each trace (X/4).
4. Apply the DECISION LOGIC rules based on the pass counts.
5. Return JSON in the exact format below.'''

# ── Layer 1 output format ────────────────────────────────────────────────────
OUTPUT_FORMAT_L1_REF = '''Return a valid JSON object with the following structure:

```json
{
  "trace_a": {
    "criteria": {
      "abduction": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "suspicion": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "depth": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "statement": {"explanation": "Brief reason why it passed/failed", "score": 1/0}
    },
    "total": x
  },
  "trace_b": {
    "criteria": {
      "abduction": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "suspicion": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "depth": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
      "statement": {"explanation": "Brief reason why it passed/failed", "score": 1/0}
    },
    "total": x
  },
  "recommendation": "TRACE_A / TRACE_B / PAIRWISE_TIEBREAKER / CORRECTION_A / CORRECTION_B",
  "reasoning": "2-3 sentences explaining the decision",
  "pairwise_winner": "TRACE_A / TRACE_B / null",
  "pairwise_reasoning": "1-2 sentences explaining pairwise selection / null",
  "failed_criteria": ["abduction", "suspicion", "depth", "statement"] or null,
  "targeted_correction": {
    "abduction": [{"player": "P#", "good_expl": "...", "evil_expl": "...", "choice": "Good|Evil"}, ...],
    "suspicion": [{"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "..."}, ...],
    "depth": 1,
    "beliefs": [{"level": 1, "player": "P#", "content": "..."}, ...],
    "statement": "1-2 sentence in-character dialogue",
    "deduction": ["P#", "P#"]
  }
}
```

Field Explanations:
- score: 1 = PASS, 0 = FAIL
- explanation: Brief explanation required for BOTH pass and fail to justify the score
- total: Integer count of passed criteria (0-4), NOT a fraction string
- recommendation: One of: "TRACE_A", "TRACE_B", "PAIRWISE_TIEBREAKER", "CORRECTION_A", "CORRECTION_B"
- pairwise_winner: Only if recommendation is PAIRWISE_TIEBREAKER (both 4/4), set to "TRACE_A" or "TRACE_B"; otherwise null
- pairwise_reasoning: Only if recommendation is PAIRWISE_TIEBREAKER, provide a brief explanation; otherwise null
- failed_criteria: Only if recommendation is CORRECTION_A or CORRECTION_B, set to the failing criteria of the CHOSEN trace as a JSON array; otherwise null
- targeted_correction: Corrects the CHOSEN trace (matching recommendation A or B). Only if failed_criteria is set; a full corrected reasoning trace object with abduction, suspicion, depth, beliefs, statement, and deduction. Otherwise null.
- Set pairwise_winner to null if recommendation is not PAIRWISE_TIEBREAKER
- Set pairwise_reasoning to null if recommendation is not PAIRWISE_TIEBREAKER
- Set failed_criteria to null if recommendation is not CORRECTION_A or CORRECTION_B
- Set targeted_correction to null if recommendation is not CORRECTION_A or CORRECTION_B
- The score integer must match the conclusion stated in the explanation

IMPORTANT: Return ONLY the JSON object, no markdown code blocks, no extra text.'''

# ── Layer 2 output format ────────────────────────────────────────────────────
OUTPUT_FORMAT_L2_REF = '''Return a valid JSON object from one of two decision paths: 1) ACCEPT, or 2) TARGETED_CORRECTION.

1. If the trace fully passes and should be accepted as-is:

```json
{
  "criteria": {
    "abduction": {"explanation": "Brief reason why it passed", "score": 1},
    "suspicion": {"explanation": "Brief reason why it passed", "score": 1},
    "depth": {"explanation": "Brief reason why it passed", "score": 1},
    "statement": {"explanation": "Brief reason why it passed", "score": 1}
  },
  "total": 4,
  "recommendation": "ACCEPT",
  "reasoning": "2-3 sentences on overall quality",
  "failed_criteria": null,
  "targeted_correction": null
}
```

2. If one or more criteria have a clear, objective failure, use TARGETED_CORRECTION:

```json
{
  "criteria": {
    "abduction": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
    "suspicion": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
    "depth": {"explanation": "Brief reason why it passed/failed", "score": 1/0},
    "statement": {"explanation": "Brief reason why it passed/failed", "score": 1/0}
  },
  "total": x,
  "recommendation": "TARGETED_CORRECTION",
  "reasoning": "2-3 sentences on overall quality and decision",
  "failed_criteria": ["abduction", "suspicion"],
  "targeted_correction": {
    "abduction": [{"player": "P#", "good_expl": "...", "evil_expl": "...", "choice": "Good|Evil"}, ...],
    "suspicion": [{"player": "P#", "level": "Minimal|Mild|Moderate|Strong|Extreme", "reason": "..."}, ...],
    "depth": 1,
    "beliefs": [{"level": 1, "player": "P#", "content": "..."}, ...],
    "statement": "1-2 sentence in-character dialogue",
    "deduction": ["P#", "P#"]
  }
}
```

Field Explanations:
- score: 1 = PASS, 0 = FAIL
- explanation: Brief explanation required for BOTH pass and fail to justify the score
- total: Integer count of passed criteria (0-4), NOT a fraction string
- recommendation: "ACCEPT" if total=4; otherwise "TARGETED_CORRECTION"
- failed_criteria: Only if recommendation is TARGETED_CORRECTION, set to the failing criteria as a JSON array; otherwise null
- targeted_correction: Full corrected trace (all 6 fields) ONLY for TARGETED_CORRECTION; otherwise null
- Set failed_criteria to null if recommendation is not TARGETED_CORRECTION
- Set targeted_correction to null if recommendation is not TARGETED_CORRECTION
- The targeted correction must fix ALL failed criteria and keep the trace internally consistent
- The score integer must match the conclusion stated in the explanation

IMPORTANT: Return ONLY the JSON object, no markdown code blocks, no extra text.'''


print(f'✓ _FRAMEWORK_LABEL: {_FRAMEWORK_LABEL}')
print(f'✓ _CRITERIA_ORDER: {_CRITERIA_ORDER}')
print(f'✓ ENABLE_RECOMMENDATION_SAFETY_OVERRIDE: {ENABLE_RECOMMENDATION_SAFETY_OVERRIDE}')
print('✓ Prompt constants defined: AVALON_RULES_REF, CRITERIA_DEFINITIONS_REF,')
print('  DECISION_LOGIC_REF, EVALUATION_PROCESS_REF, OUTPUT_FORMAT_L1_REF, OUTPUT_FORMAT_L2_REF')


✓ _FRAMEWORK_LABEL: {'TRACE_A': 'ReAct', 'CORRECTION_A': 'ReAct-Claude-4.5', 'TRACE_B': 'ReCon', 'CORRECTION_B': 'ReCon-Claude-4.5'}
✓ _CRITERIA_ORDER: ['abduction', 'suspicion', 'depth', 'statement']
✓ ENABLE_RECOMMENDATION_SAFETY_OVERRIDE: False
✓ Prompt constants defined: AVALON_RULES_REF, CRITERIA_DEFINITIONS_REF,
  DECISION_LOGIC_REF, EVALUATION_PROCESS_REF, OUTPUT_FORMAT_L1_REF, OUTPUT_FORMAT_L2_REF


## Section 4b: Layer 1 — verify_trace_pair()


In [21]:
def _coerce_prompt_text(v) -> str:
    """Coerce any value into a safe UTF-8 string for API prompt payloads."""
    if v is None:
        return ''
    try:
        if pd.isna(v):
            return ''
    except Exception:
        pass

    if isinstance(v, str):
        s = v
    else:
        try:
            if isinstance(v, (dict, list, tuple, set)):
                s = json.dumps(v, ensure_ascii=False)
            else:
                s = str(v)
        except Exception:
            s = str(v)

    s = s.replace('\x00', '')
    return s.encode('utf-8', errors='replace').decode('utf-8', errors='replace')


def _build_layer1_prompt(game_id, round_label, investigator, player_roles_str,
                          is_final_str, prior_summary_gold, prior_reasoning_gold,
                          cum_hist, cum_disc, trace_a_str, trace_b_str):
    game_id = _coerce_prompt_text(game_id)
    round_label = _coerce_prompt_text(round_label)
    investigator = _coerce_prompt_text(investigator)
    player_roles_str = _coerce_prompt_text(player_roles_str)
    is_final_str = _coerce_prompt_text(is_final_str)
    prior_summary_gold = _coerce_prompt_text(prior_summary_gold)
    prior_reasoning_gold = _coerce_prompt_text(prior_reasoning_gold)
    cum_hist = _coerce_prompt_text(cum_hist)
    cum_disc = _coerce_prompt_text(cum_disc)
    trace_a_str = _coerce_prompt_text(trace_a_str)
    trace_b_str = _coerce_prompt_text(trace_b_str)

    sep = '=' * 65
    ch  = cum_hist if cum_hist.strip() else '(none)'
    cd  = cum_disc if cum_disc.strip() else '(none)'
    ps  = prior_summary_gold if prior_summary_gold.strip() \
          else '(none - this is Round 1 or prior summaries not available)'
    pg  = prior_reasoning_gold if prior_reasoning_gold.strip() \
          else '(none - this is Round 1 or prior golds not available)'
    parts = [
        'You are evaluating two AI-generated Theory of Mind reasoning traces for the same Avalon game round. Use 4 binary PASS/FAIL criteria, then apply the decision logic strictly. Judge from the Investigator\'s perspective using observable evidence only.',
        'The Investigator is always Good, always submitted PASS, and does not speak.',
        '',
        sep, 'GAME CONTEXT', sep,
        'Game ID             : ' + game_id,
        'Round               : ' + round_label,
        'Investigator        : ' + investigator + '  (always Good; always PASS; does NOT speak)',
        'Player Roles (ground truth for your reference): ' + player_roles_str,
        'Is Final Round      : ' + is_final_str,
        '',
        sep, 'AVALON GAME MECHANICS', sep,
        AVALON_RULES_REF,
        '',
        sep, 'CUMULATIVE PUBLIC HISTORY (all rounds up to ' + round_label + ')', sep,
        ch,
        '',
        sep, 'CUMULATIVE ROUND SUMMARIES (all rounds up to ' + round_label + ')', sep,
        ps,
        '',
        sep, 'CUMULATIVE DISCUSSION LOG (all rounds up to ' + round_label + ')', sep,
        cd,
        '',
        sep, 'PRIOR ROUND REASONING GOLDS (before ' + round_label + ')', sep,
        pg,
        '',
        sep, 'TRACE A', sep,
        trace_a_str,
        '',
        sep, 'TRACE B', sep,
        trace_b_str,
        '',
        sep, 'EVALUATION CRITERIA (1=PASS, 0=FAIL)', sep,
        CRITERIA_DEFINITIONS_REF,
        '',
        sep, 'DECISION LOGIC', sep,
        DECISION_LOGIC_REF,
        '',
        sep, 'EVALUATION PROCESS', sep,
        EVALUATION_PROCESS_REF,
        '',
        sep, 'REQUIRED OUTPUT FORMAT', sep,
        OUTPUT_FORMAT_L1_REF,
    ]
    return '\n'.join(parts)


def verify_trace_pair(game_id, round_id, protagonist, player_roles,
                       is_final_round, prior_summary_gold, prior_reasoning_gold,
                       cum_hist, cum_disc,
                       react_trace_str, recon_trace_str):
    """
    Layer 1: Blind pairwise comparison of ReAct (Trace A) vs ReCon (Trace B).
    Returns (result_dict, raw_response) or (None, error_str) on failure.
    """
    round_label  = str(round_id) if str(round_id).upper().startswith('R') else 'R' + str(round_id)
    is_final_str = 'YES (deduction must contain exactly 2 players)' if is_final_round else 'NO'

    prompt = _build_layer1_prompt(
        game_id, round_label, protagonist, str(player_roles), is_final_str,
        prior_summary_gold, prior_reasoning_gold,
        cum_hist, cum_disc, react_trace_str, recon_trace_str
    )
    if _PRINT_PROMPT_DEBUG['layer1']:
        print('=== Layer 1 Prompt ===')
        print(prompt)
        print('=' * 40)

    _response_text = None
    for _retry in range(MAX_API_RETRIES):
        try:
            response = client.chat.completions.create(
                model=VERIFIER_MODEL,
                max_tokens=8192,
                messages=[
                    {'role': 'system', 'content': _coerce_prompt_text(VERIFIER_SYSTEM_PROMPT)},
                    {'role': 'user',   'content': _coerce_prompt_text(prompt)},
                ],
            )
            _response_text = response.choices[0].message.content
            break
        except Exception as _e:
            if _retry < MAX_API_RETRIES - 1:
                _delay = 5 * (2 ** _retry) + random.uniform(0, 2)
                print(f'  API error (L1, attempt {_retry+1}/{MAX_API_RETRIES}): '
                      f'{str(_e)[:80]}. Retrying in {_delay:.1f}s...')
                time.sleep(_delay)
            else:
                return None, str(_e)

    if _response_text is None:
        return None, 'No response after retries'

    try:
        clean = _response_text.strip()
        if clean.startswith('```'):
            lines = clean.split('\n')
            lines = lines[1:]
            while lines and lines[-1].strip().startswith('```'):
                lines = lines[:-1]
            clean = '\n'.join(lines)

        try:
            data = json.loads(clean)
        except json.JSONDecodeError:
            data = ast.literal_eval(clean)

        def _to_binary_score(v, lbl):
            try:
                s = int(v)
            except Exception:
                raise ValueError(f'{lbl} score is not int-like: {v}')
            if s not in (0, 1):
                raise ValueError(f'{lbl} score must be 0/1, got {s}')
            return s

        def _extract_trace_block(block, block_name):
            if not isinstance(block, dict):
                raise ValueError(f'{block_name} must be an object')
            crit = block.get('criteria')
            if not isinstance(crit, dict):
                raise ValueError(f'{block_name}.criteria must be an object')
            scores, expls = {}, {}
            for c in _CRITERIA_ORDER:
                cobj = crit.get(c)
                if not isinstance(cobj, dict):
                    raise ValueError(f'{block_name}.criteria.{c} missing/invalid')
                scores[c] = _to_binary_score(cobj.get('score'), f'{block_name}.{c}')
                expls[c]  = str(cobj.get('explanation', ''))
            total = block.get('total', sum(scores.values()))
            if isinstance(total, str):
                total = int(str(total).split('/')[0])
            else:
                total = int(total)
            if total < 0 or total > len(_CRITERIA_ORDER):
                raise ValueError(f'{block_name}.total out of range: {total}')
            return scores, expls, total

        a_scores, a_expls, a_total = _extract_trace_block(data.get('trace_a'), 'trace_a')
        b_scores, b_expls, b_total = _extract_trace_block(data.get('trace_b'), 'trace_b')

        rec = str(data.get('recommendation', 'TRACE_A')).strip().upper()
        valid_recs = {'TRACE_A', 'TRACE_B', 'PAIRWISE_TIEBREAKER', 'CORRECTION_A', 'CORRECTION_B'}
        if rec not in valid_recs:
            raise ValueError(f'Invalid recommendation: {rec}')

        if ENABLE_RECOMMENDATION_SAFETY_OVERRIDE and rec in ('CORRECTION_A', 'CORRECTION_B'):
            if a_total == 4 and b_total == 4:
                rec = 'PAIRWISE_TIEBREAKER'
            elif a_total == 4:
                rec = 'TRACE_A'
            elif b_total == 4:
                rec = 'TRACE_B'

        pw  = data.get('pairwise_winner')
        pwr = data.get('pairwise_reasoning')

        # Normalize failed_criteria to a list (Claude may return string or array)
        fc_raw = data.get('failed_criteria')
        if isinstance(fc_raw, str):
            fc = [fc_raw] if fc_raw else None
        elif isinstance(fc_raw, list):
            fc = fc_raw if fc_raw else None
        else:
            fc = None

        tc  = data.get('targeted_correction')

        needs_retry = False
        retry_reason = None

        if rec == 'PAIRWISE_TIEBREAKER':
            if pw not in ('TRACE_A', 'TRACE_B'):
                needs_retry = True
                retry_reason = 'Pairwise_Tiebreaker'
                pw = 'TRACE_A' if a_total >= b_total else 'TRACE_B'
        else:
            pw = None
            pwr = None

        if rec in ('CORRECTION_A', 'CORRECTION_B'):
            if not fc or not all(c in _CRITERIA_ORDER for c in fc):
                needs_retry = True
                retry_reason = 'Targeted'
            if not validate_full_trace_object(tc):
                needs_retry = True
                retry_reason = 'Targeted'
        else:
            fc = None
            tc = None

        result = {
            'react_criteria':              a_scores,
            'react_criteria_explanations': a_expls,
            'recon_criteria':              b_scores,
            'recon_criteria_explanations': b_expls,
            'react_total':         a_total,
            'recon_total':         b_total,
            'recommendation':      rec,
            'reasoning':           data.get('reasoning', ''),
            'pairwise_winner':     pw,
            'pairwise_reasoning':  pwr,
            'failed_criteria':     fc,
            'targeted_correction': tc,
            'correction_mode':     None,
            'needs_retry':         needs_retry,
            'retry_reason':        retry_reason,
        }
        return result, _response_text

    except Exception as _pe:
        return None, f'Parse error: {str(_pe)} | Response[:300]: {_response_text[:300]}'


print('✓ verify_trace_pair() defined (Layer 1 pairwise comparison)')


✓ verify_trace_pair() defined (Layer 1 pairwise comparison)


## Section 5: Layer 2 — verify_single_trace()


In [22]:
def _coerce_prompt_text(v) -> str:
    """Coerce any value into a safe UTF-8 string for API prompt payloads."""
    if v is None:
        return ''
    try:
        if pd.isna(v):
            return ''
    except Exception:
        pass

    if isinstance(v, str):
        s = v
    else:
        try:
            if isinstance(v, (dict, list, tuple, set)):
                s = json.dumps(v, ensure_ascii=False)
            else:
                s = str(v)
        except Exception:
            s = str(v)

    s = s.replace('\x00', '')
    return s.encode('utf-8', errors='replace').decode('utf-8', errors='replace')


def _build_layer2_prompt(game_id, round_label, investigator, player_roles_str,
                          is_final_str, prior_summary_gold, prior_reasoning_gold,
                          cum_hist, cum_disc, trace_str):
    game_id = _coerce_prompt_text(game_id)
    round_label = _coerce_prompt_text(round_label)
    investigator = _coerce_prompt_text(investigator)
    player_roles_str = _coerce_prompt_text(player_roles_str)
    is_final_str = _coerce_prompt_text(is_final_str)
    prior_summary_gold = _coerce_prompt_text(prior_summary_gold)
    prior_reasoning_gold = _coerce_prompt_text(prior_reasoning_gold)
    cum_hist = _coerce_prompt_text(cum_hist)
    cum_disc = _coerce_prompt_text(cum_disc)
    trace_str = _coerce_prompt_text(trace_str)

    sep = '=' * 65
    ch  = cum_hist if cum_hist.strip() else '(none)'
    cd  = cum_disc if cum_disc.strip() else '(none)'
    ps  = prior_summary_gold if prior_summary_gold.strip() else '(none)'
    pg  = prior_reasoning_gold if prior_reasoning_gold.strip() else '(none)'
    parts = [
        'You are re-evaluating a single AI-generated Theory of Mind reasoning trace for an Avalon game round. This trace has been selected as the best candidate and received a targeted correction.',
        '',
        'Round summaries (including current round) and prior reasoning golds (previous rounds only) are provided below when available.',
        '',
        'Re-score the trace on the same 4 binary criteria (abduction plausibility, suspicion level consistency, gated depth reasoning, statement and deduction coherence). Evaluate from the Investigator\'s perspective: assess naturalness given OBSERVABLE EVIDENCE ONLY, not factual correctness about actual alignments.',
        'The Investigator is ALWAYS Good and ALWAYS submitted PASS on every quest.',
        '',
        sep, 'GAME CONTEXT', sep,
        'Game ID             : ' + game_id,
        'Round               : ' + round_label,
        'Investigator        : ' + investigator + '  (always Good; always PASS; does NOT speak)',
        'Player Roles (ground truth for your reference): ' + player_roles_str,
        'Is Final Round      : ' + is_final_str,
        '',
        sep, 'AVALON MECHANICS REFERENCE', sep,
        AVALON_RULES_REF,
        '',
        sep, 'CUMULATIVE PUBLIC HISTORY', sep,
        ch,
        '',
        sep, 'CUMULATIVE ROUND SUMMARIES (all rounds up to ' + round_label + ')', sep,
        ps,
        '',
        sep, 'CUMULATIVE DISCUSSION LOG', sep,
        cd,
        '',
        sep, 'PRIOR ROUND REASONING GOLDS (before ' + round_label + ')', sep,
        pg,
        '',
        sep, 'TRACE TO EVALUATE', sep,
        trace_str,
        '',
        sep, 'EVALUATION CRITERIA (binary 1=PASS, 0=FAIL)', sep,
        CRITERIA_DEFINITIONS_REF,
        '',
        sep, 'OUTPUT FORMAT', sep,
        OUTPUT_FORMAT_L2_REF,
    ]
    return '\n'.join(parts)


def verify_single_trace(game_id, round_id, protagonist, player_roles,
                         is_final_round, prior_summary_gold, prior_reasoning_gold,
                         cum_hist, cum_disc, trace_str):
    """
    Layer 2: Re-evaluate a single (post-correction) trace.
    Returns (result_dict, raw_response) or (None, error_str) on failure.
    """
    round_label  = str(round_id) if str(round_id).upper().startswith('R') else 'R' + str(round_id)
    is_final_str = 'YES (deduction must contain exactly 2 players)' if is_final_round else 'NO'

    prompt = _build_layer2_prompt(
        game_id, round_label, protagonist, str(player_roles), is_final_str,
        prior_summary_gold, prior_reasoning_gold,
        cum_hist, cum_disc, trace_str
    )
    if _PRINT_PROMPT_DEBUG['layer2']:
        print('=== Layer 2 Prompt ===')
        print(prompt)
        print('=' * 40)

    _response_text = None
    for _retry in range(MAX_API_RETRIES):
        try:
            response = client.chat.completions.create(
                model=VERIFIER_MODEL,
                max_tokens=4096,
                messages=[
                    {'role': 'system', 'content': _coerce_prompt_text(VERIFIER_SYSTEM_PROMPT)},
                    {'role': 'user',   'content': _coerce_prompt_text(prompt)},
                ],
            )
            _response_text = response.choices[0].message.content
            break
        except Exception as _e:
            if _retry < MAX_API_RETRIES - 1:
                _delay = 5 * (2 ** _retry) + random.uniform(0, 2)
                print(f'  API error (L2, attempt {_retry+1}/{MAX_API_RETRIES}): '
                      f'{str(_e)[:80]}. Retrying in {_delay:.1f}s...')
                time.sleep(_delay)
            else:
                return None, str(_e)

    if _response_text is None:
        return None, 'No response after retries'

    try:
        clean = _response_text.strip()
        if clean.startswith('```'):
            lines = clean.split('\n')
            lines = lines[1:]
            while lines and lines[-1].strip().startswith('```'):
                lines = lines[:-1]
            clean = '\n'.join(lines)

        try:
            data = json.loads(clean)
        except json.JSONDecodeError:
            data = ast.literal_eval(clean)

        crit = data.get('criteria')
        if not isinstance(crit, dict):
            raise ValueError('criteria must be an object')

        scores = {}
        expls = {}
        for c in _CRITERIA_ORDER:
            cobj = crit.get(c)
            if not isinstance(cobj, dict):
                raise ValueError(f'criteria.{c} missing/invalid')
            sval = int(cobj.get('score', 0))
            if sval not in (0, 1):
                raise ValueError(f'criteria.{c}.score must be 0/1, got {sval}')
            scores[c] = sval
            expls[c]  = str(cobj.get('explanation', ''))

        total = data.get('total', sum(scores.values()))
        if isinstance(total, str):
            total = int(str(total).split('/')[0])
        else:
            total = int(total)

        rec = str(data.get('recommendation', 'ACCEPT' if total == 4 else 'TARGETED_CORRECTION')).strip().upper()
        if rec not in ('ACCEPT', 'TARGETED_CORRECTION'):
            rec = 'ACCEPT' if total == 4 else 'TARGETED_CORRECTION'

        tc = data.get('targeted_correction')
        tc_ok = validate_full_trace_object(tc)
        needs_retry = (rec == 'TARGETED_CORRECTION' and not tc_ok)

        # Normalize failed_criteria to a list
        fc_raw = data.get('failed_criteria')
        if isinstance(fc_raw, str):
            fc = [fc_raw] if fc_raw else None
        elif isinstance(fc_raw, list):
            fc = fc_raw if fc_raw else None
        else:
            fc = None

        result = {
            'total':                  total,
            'criteria':               scores,
            'criteria_explanations':  expls,
            'recommendation':         rec,
            'reasoning':              data.get('reasoning', ''),
            'failed_criteria':        fc if rec == 'TARGETED_CORRECTION' else None,
            'targeted_correction':    tc if rec == 'TARGETED_CORRECTION' else None,
            'needs_retry':            needs_retry,
        }

        # Determine the trace string to use after recheck
        if rec == 'TARGETED_CORRECTION' and tc_ok:
            result['selected_trace'] = format_trace_json(tc)
        else:
            result['selected_trace'] = trace_str

        return result, _response_text

    except Exception as _pe:
        return None, f'Parse error: {str(_pe)} | Response[:300]: {_response_text[:300]}'


print('✓ verify_single_trace() defined (Layer 2 single-trace recheck)')


✓ verify_single_trace() defined (Layer 2 single-trace recheck)


## Section 6: Incremental Save Helpers


In [ ]:
def _flush_results(vlist, clist):
    """Upsert verified and criteria records to their respective CSV files."""
    if not vlist:
        return

    _vdf = pd.DataFrame(vlist)
    _cdf = pd.DataFrame(clist)

    # Defensive dedup within vlist — last write wins (e.g. retry overwrites fallback)
    _vdf = _vdf.drop_duplicates(subset=['game_id', 'round_id'], keep='last')
    _cdf = _cdf.drop_duplicates(subset=['ID'], keep='last')

    # ── Upsert verified CSV ─────────────────────────────────────────────────
    _done_keys = set(zip(_vdf['game_id'].astype(str), _vdf['round_id'].astype(str)))
    if OUTPUT_VERIFIED_CSV.exists():
        _old = pd.read_csv(OUTPUT_VERIFIED_CSV)
        _mask = _old.apply(
            lambda r: (str(r['game_id']), str(r['round_id'])) not in _done_keys, axis=1
        )
        _old = _old[_mask]
        _vdf = pd.concat([_old, _vdf], ignore_index=True)
    _vdf['_gn'] = _vdf['game_id'].str.replace('G', '', regex=False).astype(int)
    _vdf['_rn'] = _vdf['round_id'].astype(str).str.replace('R', '', regex=False).astype(int)
    _vdf = _vdf.sort_values(['_gn', '_rn']).drop(columns=['_gn', '_rn'])
    _vdf.to_csv(OUTPUT_VERIFIED_CSV, index=False)

    # ── Upsert criteria CSV ─────────────────────────────────────────────────
    _done_ids = set(_cdf['ID'])
    if OUTPUT_CRITERIA_CSV.exists():
        _oldc = pd.read_csv(OUTPUT_CRITERIA_CSV)
        _oldc = _oldc[~_oldc['ID'].isin(_done_ids)]
        _cdf  = pd.concat([_oldc, _cdf], ignore_index=True)
    _cdf.to_csv(OUTPUT_CRITERIA_CSV, index=False)

    print(f'  \U0001f4be Flushed {len(vlist)} rows → '
          f'{OUTPUT_VERIFIED_CSV.name} + {OUTPUT_CRITERIA_CSV.name}')


print('✓ _flush_results() defined (row-level upsert, dedup-safe)')


✓ _flush_results() defined (row-level upsert)


## Step 7a: Main Verification Loop

Processes all rows in the reasoning CSVs (or up to `TEST_LIMIT`).

For each row:
1. Skip if already verified (from a prior run)
2. Build cumulative public history and discussion log
3. **Layer 1**: blind pairwise comparison → select winner or apply targeted correction
4. **Layer 2** (only when correction applied): inline recheck up to 3 attempts
5. Track results; flush to CSV every `SAVE_EVERY_N_ROWS` rows

**Resumable**: Re-running this cell picks up where it left off.


In [ ]:
# ── Initialise accumulators ──────────────────────────────────────────────────
verified_results = []
criteria_results = []
retry_rows       = []
verification_stats = {
    'total_rows': 0, 'react_selected': 0, 'recon_selected': 0,
    'react_corrected': 0, 'recon_corrected': 0,
    'pairwise_tiebreaker': 0, 'needs_human': 0, 'errors': 0,
}

# ── Load already-completed rows ───────────────────────────────────────────────
already_done = set()
if OUTPUT_VERIFIED_CSV.exists():
    _ev = pd.read_csv(OUTPUT_VERIFIED_CSV)
    for _, _er in _ev.iterrows():
        already_done.add((str(_er['game_id']), str(_er['round_id'])))
    print(f'Resuming: {len(already_done)} already-verified rows found in '
          f'{OUTPUT_VERIFIED_CSV.name}')
else:
    print('No existing output CSV — starting fresh')

rows_since_flush = 0
_total = len(react_df) if TEST_LIMIT is None else min(TEST_LIMIT, len(react_df))
print(f'Processing up to {_total} rows ({len(react_df)} total in ReAct CSV)...\n' + '='*60)

# ── Main loop ─────────────────────────────────────────────────────────────────
for idx in range(len(react_df)):

    # Enforce TEST_LIMIT on rows actually processed
    if TEST_LIMIT is not None and verification_stats['total_rows'] >= TEST_LIMIT:
        break

    row      = react_df.iloc[idx]
    game_id  = str(row['game_id'])
    round_id = str(row['round_id'])
    _id_key  = (game_id, round_id)
    score_id = game_id + '/' + round_id

    if _id_key in already_done:
        continue   # already verified in a previous run

    protagonist  = resolve_protagonist(game_id, row)
    player_roles = '' if pd.isna(row.get('player_roles', '')) else str(row['player_roles'])

    # Build cumulative context from the ReAct CSV (same per-round data as ReCon)
    ctx      = build_cumulative_context(game_id, round_id, react_df)
    cum_hist = ctx['cumulative_public_history']
    cum_disc = ctx['cumulative_discussion_log']
    is_final = ctx['is_final_round']

    # Prior round context: cumulative summaries + reasoning golds
    prior_sum_str  = get_prior_summary_str(game_id, round_id)
    prior_gold_str = get_prior_gold_str(game_id, round_id)

    # Get raw trace strings
    react_trace_raw = row.get('reasoning_gold_react', '')
    if pd.isna(react_trace_raw): react_trace_raw = ''

    recon_row       = recon_lookup.get(_id_key)
    recon_trace_raw = ''
    if recon_row is not None:
        recon_trace_raw = recon_row.get('reasoning_gold_recon', '')
        if pd.isna(recon_trace_raw): recon_trace_raw = ''

    if not str(react_trace_raw).strip() or not str(recon_trace_raw).strip():
        print(f'[{score_id}] ⚠  Missing trace(s) — queued for retry')
        retry_rows.append({'game_id': game_id, 'round_id': round_id, 'reason': 'missing_trace'})
        continue

    # Reformat traces using the same serialiser as log-gen-reasoner.ipynb
    _rd = parse_trace_string(react_trace_raw)
    _cd = parse_trace_string(recon_trace_raw)
    react_trace_str = format_trace_json(_rd) if _rd else str(react_trace_raw)
    recon_trace_str = format_trace_json(_cd) if _cd else str(recon_trace_raw)

    print(f'\n[{idx+1}/{len(react_df)}] {score_id}  final={is_final}  '
          f'protagonist={protagonist}')

    # ── Layer 1: Pairwise comparison ─────────────────────────────────────────
    time.sleep(0.5)
    result, raw_response = verify_trace_pair(
        game_id, round_id, protagonist, player_roles,
        is_final, prior_sum_str, prior_gold_str,
        cum_hist, cum_disc, react_trace_str, recon_trace_str
    )

    if result is None:
        print(f'  ✗ L1 API error: {str(raw_response)[:80]}')
        verification_stats['errors'] += 1
        retry_rows.append({'game_id': game_id, 'round_id': round_id, 'reason': 'api_error_l1'})
        continue

    print(f'  ReAct: {result["react_total"]}/4  |  ReCon: {result["recon_total"]}/4  '
          f'|  Rec: {result["recommendation"]}')

    # ── Determine selected trace and framework label ──────────────────────────
    rec = result['recommendation']

    if rec == 'TRACE_A':
        framework_used = 'ReAct'
        selected_trace = react_trace_str
        result['correction_mode'] = None
        verification_stats['react_selected'] += 1

    elif rec == 'TRACE_B':
        framework_used = 'ReCon'
        selected_trace = recon_trace_str
        result['correction_mode'] = None
        verification_stats['recon_selected'] += 1

    elif rec == 'PAIRWISE_TIEBREAKER':
        pw = result.get('pairwise_winner') or 'TRACE_A'
        if pw == 'TRACE_B':
            framework_used, selected_trace = 'ReCon', recon_trace_str
        else:
            framework_used, selected_trace = 'ReAct', react_trace_str
        result['correction_mode'] = 'Pairwise_Tiebreaker'
        verification_stats['pairwise_tiebreaker'] += 1

    elif rec in ('CORRECTION_A', 'CORRECTION_B'):
        _base_is_a  = (rec == 'CORRECTION_A')
        _base_trace = react_trace_str if _base_is_a else recon_trace_str
        _base_fw    = 'ReAct' if _base_is_a else 'ReCon'
        result['correction_mode'] = 'Targeted'

        tc = result.get('targeted_correction')
        if validate_full_trace_object(tc) and not result.get('needs_retry', False):
            selected_trace = format_trace_json(tc)
            framework_used = _base_fw + '-Claude-4.5'
            # Count as corrected only when the correction actually succeeded
            if _base_is_a:
                verification_stats['react_corrected'] += 1
            else:
                verification_stats['recon_corrected'] += 1
        else:
            print('  ⚠  targeted_correction invalid/missing — using base trace as-is')
            result['needs_retry'] = True
            result['retry_reason'] = 'Targeted'
            selected_trace = _base_trace
            framework_used = _base_fw

    else:
        print(f'  ⚠  Unexpected recommendation "{rec}" — defaulting to ReAct')
        framework_used = 'ReAct'
        selected_trace = react_trace_str
        result['correction_mode'] = None
        verification_stats['react_selected'] += 1

    # ── Layer 2: Inline recheck (only for Targeted corrections) ──────────────
    _inl_mode     = 'N/A'
    _inl_score    = 'N/A'
    _inl_rsn      = 'N/A'
    _inl_expls    = 'N/A'
    _inl_attempts = 'N/A'

    if result['correction_mode'] == 'Targeted' and '-Claude-4.5' in framework_used:
        _inl_trace = selected_trace
        _inl_accepted = False

        for _att in range(1, 4):
            time.sleep(0.5)
            print(f'  🔍 Inline L2 recheck attempt {_att}/3...')
            _vst, _vst_raw = verify_single_trace(
                game_id, round_id, protagonist, player_roles,
                is_final, prior_sum_str, prior_gold_str,
                cum_hist, cum_disc, _inl_trace
            )

            if _vst is None:
                print(f'  ⚠  L2 API error (attempt {_att}): {str(_vst_raw)[:60]}')
                if _att == 3:
                    framework_used = framework_used + '-NEEDS_HUMAN'
                    _inl_mode, _inl_score = 'NEEDS_HUMAN', 'API_ERROR'
                    _inl_rsn   = 'API error on all 3 inline attempts'
                    _inl_expls = '{}'
                    verification_stats['needs_human'] += 1
                continue

            _inl_tot   = _vst['total']
            _inl_score = str(_inl_tot) + ' of 4'
            _inl_rsn   = _vst.get('reasoning', '')
            _inl_expls = json.dumps(_vst.get('criteria_explanations', {}))

            if _inl_tot == 4:
                print(f'  ✓ L2: 4/4 — accepted (attempt {_att})')
                _inl_trace     = _vst.get('selected_trace') or _inl_trace
                selected_trace = _inl_trace
                _inl_mode      = 'INLINE_ACCEPTED' if _att == 1 else 'INLINE_FIXED'
                _inl_accepted  = True
                break

            if _vst.get('needs_retry', False):
                print('  ⚠  L2 targeted_correction missing/invalid — retrying')
                _inl_mode = 'INLINE_FIXED'
                if _att == 3:
                    print('  ✗ L2: 3 attempts failed — flagging NEEDS_HUMAN')
                    framework_used = framework_used + '-NEEDS_HUMAN'
                    _inl_mode = 'NEEDS_HUMAN'
                    verification_stats['needs_human'] += 1
                continue

            print(f'  ⚠  L2: {_inl_tot}/4 — applying fix, retrying (attempt {_att})')
            if validate_full_trace_object(_vst.get('targeted_correction')):
                _inl_trace = format_trace_json(_vst['targeted_correction'])
            _inl_mode = 'INLINE_FIXED'
            if _att == 3 and not _inl_accepted:
                print('  ✗ L2: 3 attempts failed — flagging NEEDS_HUMAN')
                framework_used = framework_used + '-NEEDS_HUMAN'
                selected_trace = _inl_trace
                _inl_mode = 'NEEDS_HUMAN'
                verification_stats['needs_human'] += 1
        _inl_attempts = _att

    # ── Build output row ──────────────────────────────────────────────────────
    _pdata = {}
    for _col in PRESERVE_COLS:
        _v = row.get(_col, '')
        _pdata[_col] = '' if pd.isna(_v) else _v
    _pdata['reasoning_gold']   = selected_trace
    _pdata['framework_used']   = framework_used
    _pdata['claude_reasoning'] = result.get('reasoning', '')
    verified_results.append(_pdata)

    # ── Build criteria record ─────────────────────────────────────────────────
    criteria_results.append({
        'ID':                          score_id,
        'ReAct_Abduction':             result['react_criteria'].get('abduction', 0),
        'ReAct_Suspicion':             result['react_criteria'].get('suspicion',  0),
        'ReAct_Depth':                 result['react_criteria'].get('depth',      0),
        'ReAct_Statement':             result['react_criteria'].get('statement',  0),
        'ReAct_Total':                 str(result['react_total']) + ' of 4',
        'ReCon_Abduction':             result['recon_criteria'].get('abduction', 0),
        'ReCon_Suspicion':             result['recon_criteria'].get('suspicion',  0),
        'ReCon_Depth':                 result['recon_criteria'].get('depth',      0),
        'ReCon_Statement':             result['recon_criteria'].get('statement',  0),
        'ReCon_Total':                 str(result['recon_total']) + ' of 4',
        'Selected_Framework':          framework_used,
        'Correction_Mode':             result.get('correction_mode') or 'None',
        'Claude_Reasoning':            result.get('reasoning', ''),
        'ReAct_Criteria_Explanations': json.dumps(result.get('react_criteria_explanations', {})),
        'ReCon_Criteria_Explanations': json.dumps(result.get('recon_criteria_explanations', {})),
        'Inline_Recheck_Mode':         _inl_mode,
        'Inline_Recheck_Score':        _inl_score,
        'Inline_Attempts':             _inl_attempts,
        'Claude_Response':             raw_response,
    })

    verification_stats['total_rows'] += 1
    already_done.add(_id_key)
    rows_since_flush += 1

    # Update in-run prior gold map so subsequent rounds of this game have this gold available
    if game_id not in _prior_gold_map:
        _prior_gold_map[game_id] = {}
    _prior_gold_map[game_id][round_id] = selected_trace

    print(f'  ✓ {score_id} → {framework_used}')

    # Queue for Step 7b retry when Layer 1 returned a silent structural failure
    if result.get('needs_retry', False):
        _rr = result.get('retry_reason') or (result.get('correction_mode') or 'retry_needed')
        retry_rows.append({'game_id': game_id, 'round_id': round_id, 'reason': _rr})
        print(f'  ⚠  Queued for Step 7b retry (reason: {_rr})')

    # ── Periodic flush ────────────────────────────────────────────────────────
    if rows_since_flush >= SAVE_EVERY_N_ROWS:
        print(f'\n  Periodic flush at row {idx+1}...')
        _flush_results(verified_results, criteria_results)
        verified_results.clear()
        criteria_results.clear()
        rows_since_flush = 0

# ── Final flush ───────────────────────────────────────────────────────────────
if rows_since_flush > 0:
    print('\nFinal flush...')
    _flush_results(verified_results, criteria_results)
    verified_results.clear()
    criteria_results.clear()

print('\n' + '='*60)
print('STEP 7a COMPLETE')
print('='*60)
for k, v in verification_stats.items():
    print(f'  {k:<25s}: {v}')


No existing output CSV — starting fresh
Processing up to 2 rows (1000 total in ReAct CSV)...

[1/1000] G001/1  final=False  protagonist=P1
=== Layer 1 Prompt ===
You are evaluating two AI-generated Theory of Mind reasoning traces for the same Avalon game round. Use 4 binary PASS/FAIL criteria, then apply the decision logic strictly. Judge from the Investigator's perspective using observable evidence only.
The Investigator is always Good, always submitted PASS, and does not speak.

GAME CONTEXT
Game ID             : G001
Round               : R1
Investigator        : P1  (always Good; always PASS; does NOT speak)
Player Roles (ground truth for your reference): {"P1":"Good",
  "P2":"Good",
  "P3":"Evil",
  "P4":"Evil",
  "P5":"Good"}
Is Final Round      : NO

AVALON GAME MECHANICS
5-player game: 3 Good players, 2 Evil players. Evil players know each other.
Quest team sizes: R1=2, R2=3, R3=2, R4=3, R5=3.
All team proposals in this dataset were approved unanimously.
Quest outcomes: one FAI

## Step 7b: Retry Loop (missing traces / API errors)


In [ ]:
# Re-process rows that had missing traces, API errors, or silent correction failures in Step 7a
_retry_errors = []

for _rrow in retry_rows:
    _gid = _rrow['game_id']
    _rid = _rrow['round_id']
    _key = (_gid, _rid)
    _reason = _rrow.get('reason', 'unknown')

    # Allow retrying rows already written when the failure was structural
    if _key in already_done and _reason not in ('Targeted', 'Pairwise_Tiebreaker'):
        continue

    print(f'\nRetrying {_gid}/{_rid} (reason: {_reason})...')

    _row = react_df[(react_df['game_id'].astype(str) == _gid) &
                    (react_df['round_id'].astype(str) == _rid)]
    if _row.empty:
        print(f'  ✗ Row not found in react_df — skipping')
        _retry_errors.append(_rrow)
        continue
    _row = _row.iloc[0]

    _recon_r    = recon_lookup.get(_key)
    _react_raw  = _row.get('reasoning_gold_react', '')
    _recon_raw  = _recon_r.get('reasoning_gold_recon', '') if _recon_r is not None else ''
    if pd.isna(_react_raw): _react_raw = ''
    if pd.isna(_recon_raw): _recon_raw = ''

    if not str(_react_raw).strip() or not str(_recon_raw).strip():
        print(f'  ✗ Traces still missing — cannot process')
        _retry_errors.append(_rrow)
        continue

    _protagonist  = resolve_protagonist(_gid, _row)
    _player_roles = '' if pd.isna(_row.get('player_roles', '')) else str(_row['player_roles'])
    _ctx          = build_cumulative_context(_gid, _rid, react_df)

    # Prior round context
    _prior_sum_str  = get_prior_summary_str(_gid, _rid)
    _prior_gold_str = get_prior_gold_str(_gid, _rid)

    _rd = parse_trace_string(_react_raw)
    _cd = parse_trace_string(_recon_raw)
    _r_str = format_trace_json(_rd) if _rd else str(_react_raw)
    _c_str = format_trace_json(_cd) if _cd else str(_recon_raw)

    time.sleep(1)
    _res, _raw = verify_trace_pair(
        _gid, _rid, _protagonist, _player_roles,
        _ctx['is_final_round'],
        _prior_sum_str, _prior_gold_str,
        _ctx['cumulative_public_history'],
        _ctx['cumulative_discussion_log'],
        _r_str, _c_str
    )

    if _res is None:
        print(f'  ✗ API error on retry: {str(_raw)[:80]}')
        _retry_errors.append(_rrow)
        verification_stats['errors'] += 1
        continue

    rec = _res['recommendation']

    if rec == 'TRACE_A':
        _fw = 'ReAct'
        _st = _r_str
        _res['correction_mode'] = None

    elif rec == 'TRACE_B':
        _fw = 'ReCon'
        _st = _c_str
        _res['correction_mode'] = None

    elif rec == 'PAIRWISE_TIEBREAKER':
        _pw = _res.get('pairwise_winner') or 'TRACE_A'
        if _pw == 'TRACE_B':
            _fw, _st = 'ReCon', _c_str
        else:
            _fw, _st = 'ReAct', _r_str
        _res['correction_mode'] = 'Pairwise_Tiebreaker'

    elif rec in ('CORRECTION_A', 'CORRECTION_B'):
        _base_is_a = (rec == 'CORRECTION_A')
        _base_fw    = 'ReAct' if _base_is_a else 'ReCon'
        _res['correction_mode'] = 'Targeted'

        _tc = _res.get('targeted_correction')
        if validate_full_trace_object(_tc) and not _res.get('needs_retry', False):
            _st = format_trace_json(_tc)
            _fw = _base_fw + '-Claude-4.5'
        else:
            print('  ⚠  targeted_correction invalid/missing on retry')
            _retry_errors.append({'game_id': _gid, 'round_id': _rid, 'reason': 'Targeted'})
            continue

        if _base_is_a:
            verification_stats['react_corrected'] += 1
        else:
            verification_stats['recon_corrected'] += 1

    else:
        print(f'  ⚠  Unexpected recommendation "{rec}" on retry')
        _retry_errors.append({'game_id': _gid, 'round_id': _rid, 'reason': 'unexpected_recommendation'})
        continue

    # Inline Layer 2 recheck for corrected traces (same policy as Step 7a)
    _inl_mode     = 'N/A'
    _inl_score    = 'N/A'
    _inl_rsn      = 'N/A'
    _inl_expls    = 'N/A'
    _inl_attempts = 'N/A'

    if _res.get('correction_mode') == 'Targeted' and '-Claude-4.5' in _fw:
        _inl_trace = _st
        _inl_accepted = False

        for _att in range(1, 4):
            time.sleep(0.5)
            print(f'  🔍 Inline L2 recheck attempt {_att}/3...')
            _vst, _vst_raw = verify_single_trace(
                _gid, _rid, _protagonist, _player_roles,
                _ctx['is_final_round'],
                _prior_sum_str, _prior_gold_str,
                _ctx['cumulative_public_history'],
                _ctx['cumulative_discussion_log'],
                _inl_trace
            )

            if _vst is None:
                print(f'  ⚠  L2 API error (attempt {_att}): {str(_vst_raw)[:60]}')
                if _att == 3:
                    _fw = _fw + '-NEEDS_HUMAN'
                    _inl_mode, _inl_score = 'NEEDS_HUMAN', 'API_ERROR'
                    _inl_rsn = 'API error on all 3 inline attempts'
                    _inl_expls = '{}'
                    verification_stats['needs_human'] += 1
                continue

            _inl_tot   = _vst['total']
            _inl_score = str(_inl_tot) + ' of 4'
            _inl_rsn   = _vst.get('reasoning', '')
            _inl_expls = json.dumps(_vst.get('criteria_explanations', {}))

            if _inl_tot == 4:
                _inl_trace = _vst.get('selected_trace') or _inl_trace
                _st = _inl_trace
                _inl_mode = 'INLINE_ACCEPTED' if _att == 1 else 'INLINE_FIXED'
                _inl_accepted = True
                print(f'  ✓ L2: 4/4 — accepted (attempt {_att})')
                break

            if _vst.get('needs_retry', False):
                print('  ⚠  L2 targeted_correction missing/invalid — retrying')
                _inl_mode = 'INLINE_FIXED'
                if _att == 3:
                    _fw = _fw + '-NEEDS_HUMAN'
                    _inl_mode = 'NEEDS_HUMAN'
                    verification_stats['needs_human'] += 1
                continue

            print(f'  ⚠  L2: {_inl_tot}/4 — applying fix, retrying (attempt {_att})')
            if validate_full_trace_object(_vst.get('targeted_correction')):
                _inl_trace = format_trace_json(_vst['targeted_correction'])
            _inl_mode = 'INLINE_FIXED'
            if _att == 3 and not _inl_accepted:
                _fw = _fw + '-NEEDS_HUMAN'
                _st = _inl_trace
                _inl_mode = 'NEEDS_HUMAN'
                verification_stats['needs_human'] += 1
        _inl_attempts = _att

    _pdata = {}
    for _col in PRESERVE_COLS:
        _v = _row.get(_col, '')
        _pdata[_col] = '' if pd.isna(_v) else _v
    _pdata['reasoning_gold']   = _st
    _pdata['framework_used']   = _fw
    _pdata['claude_reasoning'] = _res.get('reasoning', '')
    verified_results.append(_pdata)

    criteria_results.append({
        'ID': _gid + '/' + _rid,
        'ReAct_Abduction': _res['react_criteria'].get('abduction', 0),
        'ReAct_Suspicion':  _res['react_criteria'].get('suspicion',  0),
        'ReAct_Depth':      _res['react_criteria'].get('depth',      0),
        'ReAct_Statement':  _res['react_criteria'].get('statement',  0),
        'ReAct_Total':      str(_res['react_total']) + ' of 4',
        'ReCon_Abduction':  _res['recon_criteria'].get('abduction', 0),
        'ReCon_Suspicion':  _res['recon_criteria'].get('suspicion',  0),
        'ReCon_Depth':      _res['recon_criteria'].get('depth',      0),
        'ReCon_Statement':  _res['recon_criteria'].get('statement',  0),
        'ReCon_Total':      str(_res['recon_total']) + ' of 4',
        'Selected_Framework':  _fw,
        'Correction_Mode':     _res.get('correction_mode') or 'None',
        'Claude_Reasoning':    _res.get('reasoning', ''),
        'ReAct_Criteria_Explanations': json.dumps(_res.get('react_criteria_explanations', {})),
        'ReCon_Criteria_Explanations': json.dumps(_res.get('recon_criteria_explanations', {})),
        'Inline_Recheck_Mode': _inl_mode,
        'Inline_Recheck_Score': _inl_score,
        'Inline_Attempts': _inl_attempts,
        'Claude_Response': _raw,
    })

    already_done.add(_key)
    verification_stats['total_rows'] += 1

    # Keep in-run prior gold map synchronized during retries as well
    if _gid not in _prior_gold_map:
        _prior_gold_map[_gid] = {}
    _prior_gold_map[_gid][_rid] = _st

    print(f'  ✓ Retry OK → {_fw}')

# Final flush for retry results
if verified_results:
    _flush_results(verified_results, criteria_results)

print(f'\n✓ Step 7b complete. Retry errors remaining: {len(_retry_errors)}')
if _retry_errors:
    for _e in _retry_errors:
        print(f'  - {_e}')

  💾 Flushed 1 rows → reasoning_gold_verified.csv + reasoning_gold_criteria_scores.csv

✓ Step 7b complete. Retry errors remaining: 0


## Step 8: Save Final Results and Print Summary Statistics


In [113]:
# Ensure final flush
if verified_results or criteria_results:
    _flush_results(verified_results, criteria_results)

# ── Load and display final outputs ──────────────────────────────────────────
_has_verified = OUTPUT_VERIFIED_CSV.exists()
_has_criteria = OUTPUT_CRITERIA_CSV.exists()

if _has_verified:
    _vdf_final = pd.read_csv(OUTPUT_VERIFIED_CSV)
else:
    _vdf_final = None

if _has_criteria:
    _cdf_final = pd.read_csv(OUTPUT_CRITERIA_CSV)
else:
    _cdf_final = None

if _has_verified or _has_criteria:
    print('=' * 60)
    print('FINAL OUTPUT SUMMARY')
    print('=' * 60)

    if _has_verified:
        print(f'reasoning_gold_verified.csv : {_vdf_final.shape[0]} rows × {_vdf_final.shape[1]} cols')
    else:
        print('reasoning_gold_verified.csv : MISSING')

    if _has_criteria:
        print(f'reasoning_gold_criteria_scores.csv : {_cdf_final.shape[0]} rows × {_cdf_final.shape[1]} cols')
    else:
        print('reasoning_gold_criteria_scores.csv : MISSING')

    if _has_verified:
        print(f'\nFramework distribution:')
        print(_vdf_final['framework_used'].value_counts().to_string())

        _nh = _vdf_final['framework_used'].str.endswith('-NEEDS_HUMAN', na=False).sum()
        print(f'\n⚠  NEEDS_HUMAN rows: {_nh}')
        if _nh > 0:
            _nh_rows = _vdf_final[_vdf_final['framework_used'].str.endswith('-NEEDS_HUMAN', na=False)]
            print(_nh_rows[['game_id', 'round_id', 'framework_used']].to_string(index=False))

    if _has_criteria:
        print(f'\nCorrection mode distribution:')
        print(_cdf_final['Correction_Mode'].value_counts().to_string())

        print(f'\nInline recheck outcome distribution:')
        print(_cdf_final['Inline_Recheck_Mode'].value_counts().to_string())

else:
    print('⚠  No output CSV found — did Step 7a run successfully?')

  💾 Flushed 1 rows → reasoning_gold_verified.csv + reasoning_gold_criteria_scores.csv
FINAL OUTPUT SUMMARY
reasoning_gold_verified.csv : 1 rows × 12 cols
reasoning_gold_criteria_scores.csv : 1 rows × 20 cols

Framework distribution:
framework_used
ReAct    1

⚠  NEEDS_HUMAN rows: 0

Correction mode distribution:
Series([], )

Inline recheck outcome distribution:
Series([], )


## Step 9: Comprehensive Analysis

Per-criterion pass rates, framework comparison, correction statistics, and
pairwise agreement analysis — suitable for paper reporting.


In [ ]:
# ── Step 9.1: Load data + total record count ──────────────────────────────────
import pandas as pd
import numpy as np

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_criteria_names = ['Abduction', 'Suspicion', 'Depth', 'Statement']
_react_scores = [sum(_cdf.iloc[i]['ReAct_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_recon_scores = [sum(_cdf.iloc[i]['ReCon_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_N = len(_cdf)

print('=' * 80)
print('COMPREHENSIVE ANALYSIS: Reasoning Trace Verification Results')
print('=' * 80)
print(f'\nTotal records analysed: {_N}')
print(f'Criteria CSV         : {OUTPUT_CRITERIA_CSV.name}')
print(f'Columns              : {list(_cdf.columns)}')

In [ ]:
# ── Step 9.2: Per-criterion pass rates ───────────────────────────────────────
import pandas as pd

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_criteria_names = ['Abduction', 'Suspicion', 'Depth', 'Statement']
_N = len(_cdf)

_react_rates, _recon_rates = {}, {}
for c in _criteria_names:
    _react_rates[c] = (_cdf['ReAct_' + c].sum() / _N) * 100
    _recon_rates[c] = (_cdf['ReCon_' + c].sum() / _N) * 100

print('=' * 80)
print('1. PER-CRITERION PASS RATES')
print('=' * 80)
print('\n{:<14s}  {:>12s}  {:>12s}  {:>12s}'.format(
      'Criterion', 'ReAct %', 'ReCon %', 'Diff'))
for c in _criteria_names:
    diff = _recon_rates[c] - _react_rates[c]
    print('  {:<12s}  {:>10.2f}%  {:>10.2f}%  {:>+10.2f}%'.format(
          c, _react_rates[c], _recon_rates[c], diff))

In [ ]:
# ── Step 9.3: Average scores and score distribution ───────────────────────────
import pandas as pd
import numpy as np

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_criteria_names = ['Abduction', 'Suspicion', 'Depth', 'Statement']
_react_scores = [sum(_cdf.iloc[i]['ReAct_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_recon_scores = [sum(_cdf.iloc[i]['ReCon_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_N = len(_cdf)

print('=' * 80)
print('2. AVERAGE SCORES AND SCORE DISTRIBUTION')
print('=' * 80)
print(f'\n  ReAct avg: {np.mean(_react_scores):.3f}/4  (SD={np.std(_react_scores):.3f})')
print(f'  ReCon avg: {np.mean(_recon_scores):.3f}/4  (SD={np.std(_recon_scores):.3f})')
print()
for score in range(5):
    rc = _react_scores.count(score)
    cc = _recon_scores.count(score)
    print(f'  {score}/4  ->  ReAct: {rc:4d} ({rc/_N*100:5.1f}%)  '
          f'ReCon: {cc:4d} ({cc/_N*100:5.1f}%)')

In [ ]:
# ── Step 9.4: Framework selection breakdown ───────────────────────────────────
import pandas as pd

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_N   = len(_cdf)

print('=' * 80)
print('3. FRAMEWORK SELECTION BREAKDOWN')
print('=' * 80)
_sel_counts = _cdf['Selected_Framework'].value_counts()
for fw, cnt in _sel_counts.items():
    print(f'  {fw:<40s}: {cnt:4d}  ({cnt/_N*100:5.2f}%)')

_react_total = _cdf['Selected_Framework'].str.startswith('ReAct').sum()
_recon_total = _cdf['Selected_Framework'].str.startswith('ReCon').sum()
_needs_human = _cdf['Selected_Framework'].str.endswith('-NEEDS_HUMAN').sum()
print(f'\n  ReAct (direct + corrected): {_react_total:4d}  ({_react_total/_N*100:5.2f}%)')
print(f'  ReCon (direct + corrected): {_recon_total:4d}  ({_recon_total/_N*100:5.2f}%)')
print(f'  NEEDS_HUMAN:                {_needs_human:4d}  ({_needs_human/_N*100:5.2f}%)')

In [ ]:
# ── Step 9.5: Correction mode distribution ────────────────────────────────────
import pandas as pd

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_N   = len(_cdf)

print('=' * 80)
print('4. CORRECTION MODE DISTRIBUTION')
print('=' * 80)
for mode, cnt in _cdf['Correction_Mode'].value_counts().items():
    print(f'  {str(mode):<25s}: {cnt:4d}  ({cnt/_N*100:5.2f}%)')

In [ ]:
# ── Step 9.6: Inline Layer 2 recheck summary ─────────────────────────────────
import pandas as pd

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_N   = len(_cdf)

print('=' * 80)
print('5. INLINE LAYER 2 RECHECK SUMMARY')
print('=' * 80)
_inl_na   = ((_cdf['Inline_Recheck_Mode'] == 'N/A') | _cdf['Inline_Recheck_Mode'].isna()).sum()
_inl_trig = _N - _inl_na
print(f'\n  No recheck needed (N/A):   {_inl_na:4d}  ({_inl_na/_N*100:5.2f}%)')
print(f'  Inline recheck triggered:  {_inl_trig:4d}  ({_inl_trig/_N*100:5.2f}%)')
print()
for mode, cnt in _cdf['Inline_Recheck_Mode'].value_counts(dropna=True).items():
    print(f'    {str(mode):<25s}: {cnt:4d}  ({cnt/_N*100:5.2f}%)')

In [ ]:
# ── Step 9.7: Per-criterion inter-framework agreement ─────────────────────────
# ── Step 9.8: Pair score summary ─────────────────────────────────────────────
import pandas as pd
import numpy as np

_cdf = pd.read_csv(OUTPUT_CRITERIA_CSV)
_criteria_names = ['Abduction', 'Suspicion', 'Depth', 'Statement']
_react_scores = [sum(_cdf.iloc[i]['ReAct_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_recon_scores = [sum(_cdf.iloc[i]['ReCon_' + c] for c in _criteria_names)
                 for i in range(len(_cdf))]
_N = len(_cdf)

print('=' * 80)
print('6. PER-CRITERION INTER-FRAMEWORK AGREEMENT')
print('=' * 80)
print('\n  (both PASS or both FAIL for same game/round)')
for c in _criteria_names:
    agree  = int((_cdf['ReAct_' + c] == _cdf['ReCon_' + c]).sum())
    both_f = int(((_cdf['ReAct_' + c] == 0) & (_cdf['ReCon_' + c] == 0)).sum())
    both_p = int(((_cdf['ReAct_' + c] == 1) & (_cdf['ReCon_' + c] == 1)).sum())
    print(f'  {c:<14s}: {agree:4d} agreements ({agree/_N*100:5.1f}%)  '
          f'both-pass: {both_p}  both-fail: {both_f}')

_both_perfect = sum(r == 4 and c == 4
                    for r, c in zip(_react_scores, _recon_scores))
_one_perfect  = sum((r == 4) != (c == 4)
                    for r, c in zip(_react_scores, _recon_scores))
_both_low     = sum(r <= 2 and c <= 2
                    for r, c in zip(_react_scores, _recon_scores))

print('\n' + '=' * 80)
print('7. PAIR SCORE SUMMARY')
print('=' * 80)
print(f'\n  Both 4/4 (pairwise tiebreaker): {_both_perfect:4d}  ({_both_perfect/_N*100:5.2f}%)')
print(f'  Exactly one 4/4:                {_one_perfect:4d}  ({_one_perfect/_N*100:5.2f}%)')
print(f'  Both <=2/4:                     {_both_low:4d}  ({_both_low/_N*100:5.2f}%)')

print('\n' + '=' * 80)
print('ANALYSIS COMPLETE')
print('=' * 80)